In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming Process Files")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .master("local[2]")
    .getOrCreate()
)
spark

In [2]:
# To allow automatic shcemaInference while reading
spark.conf.set("spark.sql.streaming.schemaInference", True)


# Create the streaming_df to read form input directory
# option cleanSource has 3 modes = Off(bydefault), DELETE and ARCHIVE

streaming_df = (
    spark.readStream
    .option("cleanSource", "archive")
    .option("sourceARchiveDir", "archive_dir")
    # this option makes spark process 1 file at a time
    .option("maxFilesPerTrigger", 1)
    .format("json")
    .load("data/input/device_files/")
)



In [3]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [4]:
# lets explode the data as devices contains list/array of device reading

from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices")).drop("data")

In [5]:
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [6]:
# Flatten the exploded df
from pyspark.sql.functions import col
flattened_df = (
    exploded_df
    .withColumn("deviceID", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [7]:
flattened_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceID: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [8]:
# Write the output to console sink to check the output

# flattened_df.writeStream.format("console").outputMode("append").start().awaitTermination()

In [ ]:
# Write the output to the a csv sink

(
    flattened_df
    .writeStream
    .format("csv")
    .outputMode("append")
    .option("path", "data/output/device_data.csv")
 .option("checkpointLocation", "checkpoint_dir")
 .start()
 .awaitTermination()
)